# TP MLOps II — Comparación y tuning de modelos

Continuación de `04_feature_engineering.ipynb`. Ahí quedó establecido que sin el atajo (`total load forecast`), **Random Forest** era el mejor modelo (MAE 1751.3 MW / MAPE 6.04%), con Gradient Boosting cerca (MAE 1793.9 MW / MAPE 6.22%).

**Objetivo de este notebook:** llevar esa comparación un paso más allá — tunear hiperparámetros de forma sistemática con `RandomizedSearchCV` + `TimeSeriesSplit`, y sumar un tercer contendiente (**LightGBM**), para practicar un workflow real de selección y ajuste de modelos antes de automatizarlo en el DAG de entrenamiento.

## 1. Datos y features (igual que en 04, sin el atajo)

Se repite la preparación del notebook anterior: features cíclicas (sin/cos de hour/dow/month), lags de demanda (t-24h, t-168h), y **sin** `total load forecast` como feature (se descartó porque era casi un atajo directo a la respuesta, ver `04_feature_engineering.ipynb`).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("../data")
df = pd.read_parquet(data_path / 'processed' / 'energy_weather_clean.parquet')

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
df['load_lag_24h'] = df['total load actual'].shift(24)
df['load_lag_168h'] = df['total load actual'].shift(168)

df_model = df.dropna(subset=['load_lag_24h', 'load_lag_168h'])

train = df_model[df_model.index.year < 2018]
test = df_model[df_model.index.year >= 2018]

feature_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
                 'temp_Madrid', 'temp_Barcelona', 'temp_Valencia', 'temp_Seville', 'temp_Bilbao',
                 'load_lag_24h', 'load_lag_168h']
target_col = 'total load actual'

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

X_train.shape, X_test.shape

((26137, 14), (8759, 14))

## 2. `TimeSeriesSplit`: validación cruzada sin mirar al futuro

Para tunear hiperparámetros hace falta medir qué tan buena es cada combinación *sin* tocar el conjunto de test (2018) — eso sería "espiar" el test y volver el resultado final poco honesto. La solución es generar validaciones **dentro de train**, respetando el orden temporal: un `KFold` común mezclaría al azar, permitiendo entrenar con datos futuros y evaluar en el pasado (leakage temporal). `TimeSeriesSplit` en cambio arma folds donde el train de cada fold es siempre anterior a su validación.

Con 5 splits, el train de cada fold crece progresivamente y el conjunto de validación (~6 meses) siempre queda inmediatamente después.

In [2]:
from sklearn.model_selection import TimeSeriesSplit 

tscv = TimeSeriesSplit(n_splits=5)

for i, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f"Fold {i}: train {len(train_idx)} filas | val {len(val_idx)} filas | "
          f"train hasta {X_train.index[train_idx[-1]]} | val hasta {X_train.index[val_idx[-1]]}")

Fold 0: train 4357 filas | val 4356 filas | train hasta 2015-07-08 11:00:00+00:00 | val hasta 2016-01-05 23:00:00+00:00
Fold 1: train 8713 filas | val 4356 filas | train hasta 2016-01-05 23:00:00+00:00 | val hasta 2016-07-05 11:00:00+00:00
Fold 2: train 13069 filas | val 4356 filas | train hasta 2016-07-05 11:00:00+00:00 | val hasta 2017-01-02 23:00:00+00:00
Fold 3: train 17425 filas | val 4356 filas | train hasta 2017-01-02 23:00:00+00:00 | val hasta 2017-07-03 11:00:00+00:00
Fold 4: train 21781 filas | val 4356 filas | train hasta 2017-07-03 11:00:00+00:00 | val hasta 2017-12-31 23:00:00+00:00


Confirmado: cada fold entrena con todo el pasado acumulado hasta cierto punto y valida con el semestre siguiente, sin nunca ver el futuro respecto a su propio train.

## 3. Random Forest — tuning con `RandomizedSearchCV`

En vez de probar manualmente combinaciones de hiperparámetros, se usa `RandomizedSearchCV`: prueba `n_iter=20` combinaciones al azar sobre una grilla de valores (`n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`), evaluando cada una con los 5 folds de `TimeSeriesSplit` (`cv=tscv`) y usando `neg_mean_absolute_error` como métrica (sklearn maximiza por convención, así que se usa el MAE negativo para poder minimizar el error real).

Al final se queda con la combinación de mejor MAE promedio entre los 5 folds.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

param_dist_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [8, 12, 15, 20, None],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 0.5, 0.8, 1.0],
}

search_rf = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=20,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    random_state=42,
    verbose=1,
    n_jobs=-1,
)

search_rf.fit(X_train, y_train)

print("Mejores parámetros RF:", search_rf.best_params_)
print("Mejor MAE (CV):", -search_rf.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejores parámetros RF: {'n_estimators': 300, 'min_samples_leaf': 10, 'max_features': 0.8, 'max_depth': 12}
Mejor MAE (CV): 1772.4364237029142


## 4. Random Forest tuneado — evaluación en test

`search_rf.best_estimator_` ya viene reentrenado sobre todo `X_train` con los mejores hiperparámetros encontrados. Se evalúa una única vez sobre `test` (2018), que no se tocó en ningún momento del tuning.

In [4]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error

best_rf = search_rf.best_estimator_ # Save the best Random Forest model
y_pred_rf_tuned = best_rf.predict(X_test) # Make predictions on the test set

mae_rf_tuned = mean_absolute_error(y_test, y_pred_rf_tuned)
mape_rf_tuned = mean_absolute_percentage_error(y_test, y_pred_rf_tuned)
rmse_rf_tuned = root_mean_squared_error(y_test, y_pred_rf_tuned)

print(f"RF tuneado:")
print(f"MAE: {mae_rf_tuned:.1f} MW")
print(f"MAPE: {mape_rf_tuned:.2%}")
print(f"RMSE: {rmse_rf_tuned:.1f} MW")

RF tuneado:
MAE: 1736.8 MW
MAPE: 6.00%
RMSE: 2534.7 MW


Mejora leve pero consistente sobre el RF sin tunear de `04` (MAE 1751.3 → 1736.8 MW; MAPE 6.04% → 6.00%). El tuning no hace magia, pero sí un ajuste fino genuino.

## 5. Gradient Boosting — tuning

Mismo procedimiento (`RandomizedSearchCV` + `TimeSeriesSplit`), grilla adaptada a boosting: `learning_rate`, `subsample`, `max_depth` más chico (los árboles de boosting son más débiles individualmente que los de un bosque, por diseño). A diferencia de Random Forest, Gradient Boosting de sklearn es secuencial (cada árbol corrige al anterior) y no paraleliza internamente, por lo que esta búsqueda tarda más.

In [5]:
from sklearn.ensemble import GradientBoostingRegressor

param_dist_gb = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
}

search_gb = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=20,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    random_state=42,
    verbose=1,
    n_jobs=-1,
)

search_gb.fit(X_train, y_train)

print("Mejores parámetros GB:", search_gb.best_params_)
print("Mejor MAE (CV):", -search_gb.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejores parámetros GB: {'subsample': 0.6, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.01}
Mejor MAE (CV): 1818.6848448473884


## 6. Gradient Boosting tuneado — evaluación en test

In [6]:
best_gb = search_gb.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test)

mae_gb_tuned = mean_absolute_error(y_test, y_pred_gb_tuned)
mape_gb_tuned = mean_absolute_percentage_error(y_test, y_pred_gb_tuned)
rmse_gb_tuned = root_mean_squared_error(y_test, y_pred_gb_tuned)

print(f"GB tuneado: MAE {mae_gb_tuned:.1f} MW | MAPE {mape_gb_tuned:.2%} | RMSE {rmse_gb_tuned:.1f} MW")


GB tuneado: MAE 1786.5 MW | MAPE 6.19% | RMSE 2565.5 MW


Mejora sobre el GB sin tunear de `04` (1793.9 → 1786.5 MW), pero queda por detrás del Random Forest tuneado (1736.8 MW).

## 7. LightGBM — tercer contendiente

Se suma LightGBM, una librería de gradient boosting optimizada (más rápida que la de sklearn, muy usada en la industria para problemas tabulares). Grilla similar a GB, con `num_leaves` como hiperparámetro extra — a diferencia de Random Forest / GB de sklearn (que crecen los árboles nivel por nivel), LightGBM crece hoja por hoja, y `num_leaves` es su forma principal de controlar la complejidad del árbol (más importante incluso que `max_depth`, que acá se deja en `-1` = sin límite en varias combinaciones).

In [7]:
from lightgbm import LGBMRegressor

param_dist_lgbm = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [-1, 4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'num_leaves': [15, 31, 63, 127],
    'subsample': [0.6, 0.8, 1.0],
}

search_lgbm = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    param_distributions=param_dist_lgbm,
    n_iter=20,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    random_state=42,
    verbose=1,
    n_jobs=-1,
)

search_lgbm.fit(X_train, y_train)

print("Mejores parámetros LGBM:", search_lgbm.best_params_)
print("Mejor MAE (CV):", -search_lgbm.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejores parámetros LGBM: {'subsample': 0.6, 'num_leaves': 15, 'n_estimators': 200, 'max_depth': -1, 'learning_rate': 0.03}
Mejor MAE (CV): 1833.822090670404


## 8. LightGBM tuneado — evaluación en test

In [8]:
best_lgbm = search_lgbm.best_estimator_
y_pred_lgbm_tuned = best_lgbm.predict(X_test)

mae_lgbm_tuned = mean_absolute_error(y_test, y_pred_lgbm_tuned)
mape_lgbm_tuned = mean_absolute_percentage_error(y_test, y_pred_lgbm_tuned)
rmse_lgbm_tuned = root_mean_squared_error(y_test, y_pred_lgbm_tuned)

print(f"LightGBM tuneado: MAE {mae_lgbm_tuned:.1f} MW | MAPE {mape_lgbm_tuned:.2%} | RMSE {rmse_lgbm_tuned:.1f} MW")


LightGBM tuneado: MAE 1794.9 MW | MAPE 6.23% | RMSE 2567.6 MW


## 9. Comparación final

| Modelo | MAE (CV) | MAE (test) | MAPE (test) |
|---|---|---|---|
| **Random Forest tuneado** | 1772.4 MW | **1736.8 MW** | **6.00%** |
| Gradient Boosting tuneado | 1818.7 MW | 1786.5 MW | 6.19% |
| LightGBM tuneado | 1833.8 MW | 1794.9 MW | 6.23% |

**Random Forest tuneado gana** con margen consistente entre validación cruzada y test (mismo ranking en ambos, lo que da confianza de que no es azar del split). Es un resultado interesante en sí: suele asumirse que el gradient boosting (LightGBM/XGBoost) le gana casi siempre a Random Forest, pero en este dataset (tamaño moderado, ~26k filas train, features relativamente simples) el bagging de Random Forest generalizó mejor que el boosting secuencial.

**Hiperparámetros ganadores (Random Forest):** `n_estimators=300, max_depth=12, min_samples_leaf=10, max_features=0.8`.

## 10. Guardado del modelo ganador

Se serializa el Random Forest tuneado con `joblib`, junto con la lista de `feature_cols` que espera — necesaria para cuando el modelo se cargue en la API más adelante, así queda explícito y reproducible qué columnas y en qué orden espera el modelo, sin depender de reconstruir el pipeline de memoria.

In [9]:
import joblib

models_path = Path('../models')
models_path.mkdir(exist_ok=True)

joblib.dump(best_rf, models_path / 'random_forest_v1.joblib')

joblib.dump(feature_cols, models_path / 'feature_cols_v1.joblib')

['..\\models\\feature_cols_v1.joblib']

## Conclusión

Modelo final del TP hasta este punto: **Random Forest** (`models/random_forest_v1.joblib`), entrenado con calendario cíclico + temperatura por ciudad + lags de demanda (sin el forecast oficial del TSO), MAE 1736.8 MW / MAPE 6.00% sobre test 2018.

**Próximo paso:** dejar de iterar en notebooks y pasar el pipeline a código Python reproducible (scripts/módulos), como base para servirlo por API y automatizarlo en Airflow.